# SoccerNet GameState Challenge

## Data Structure

data/
    SoccerNetGS/
      train/
        sequences_info.json
        SNGS-.../
            Labels-GameState.json
                {}info
                []images
                []annotations
                []categories
            img1/
                *.jpg
      valid/
        sequences_info.json
        SNGS-.../
            Labels-GameState.json
                {}info
                []images
                []annotations
                []categories
            img1/
                *.jpg
      test/
        sequences_info.json
        SNGS-.../
            Labels-GameState.json
                {}info
                []images
                []annotations
                []categories
            img1/
                *.jpg
      challenge/
        sequences_info.json
        SNGS-.../
            img1/
                *.jpg

- **sequences_info.json** is duplicated in the 4 directories. It provides to which directory each video belongs (train, valid, test, challenge). See code in next cell.
- **SNGS-... folders** ("..." represents a 3 digit number). There is one of these folders per video. *sequences_info.json* details if that video is for training, validation, testing or for the challenge itself.
- **Labels-GameState.json** describes every observation made by computer vision in each frame. Further develop in next section. Is the ground
- __*.jpg files__ are located in each *img1* directory, which contains 750 of them (750 frames of a 30 seconds video at 25 fps).

#### Print the number of folders (videos) and files in data directory (and not its children directories) and check if sequences_info.json files in different directories are the same

In [2]:
import os
from data_analysis import count_folders_and_files, check_duplicated_sequence_files, directories_path, data_directories

for directory in data_directories:
    path = os.path.join(directories_path, directory)
    folder_count, file_count, file_name = count_folders_and_files(path)
    print(f"Directory: {directory}")
    print(f"Number of folders: {folder_count}")
    print(f"Number of files: {file_count}")
    print(f"Names of those files: {file_name} \n")

print("Sequences files are the same.\n") if check_duplicated_sequence_files() else print("Sequences files are different.\n")

Directory: challenge
Number of folders: 36
Number of files: 2
Names of those files: ['.DS_Store', 'sequences_info.json'] 

Directory: test
Number of folders: 49
Number of files: 2
Names of those files: ['.DS_Store', 'sequences_info.json'] 

Directory: train
Number of folders: 57
Number of files: 2
Names of those files: ['.DS_Store', 'sequences_info.json'] 

Directory: valid
Number of folders: 58
Number of files: 2
Names of those files: ['.DS_Store', 'sequences_info.json'] 

Sequences files are the same.



### Labels-GameState.json
As briefly described above, the structure of the file is devided into 1 object (**info**) and 3 lists of objects (**images**, **annotations**, **categories**).
This file is contained in the **train**, **test** and **valid** directories but not in **challenge** as this is the ground truth of the model (the one to be compared with when evaluated).

#### { } info

> Information about the game and video.
> Except for **frame_rate** and **seq_lenght** (both number), the properties are strings.

- **version** is requested to be the latest: 1.1 (March 2024).
- **action_class** describes different states of the game, e.g. Kick-off, Offside, Corner, Shots off target, etc.
- **id** & **name** have the same number, but in **name** it is concatenated
- **im_dir** is always "img1" and represents the folder containing the 750 frames of each video.

**Example**

"info": {
    "version": "1.1",
    "game_id": "4",
    "id": "060",
    "num_tracklets": "26",
    "action_position": "895",
    "action_class": "Kick-off",
    "visibility": "visible",
    "game_time_start": "1 - 00:00",
    "game_time_stop": "1 - 00:30",
    "clip_start": "0",
    "clip_stop": "30000",
    "name": "SNGS-060",
    "im_dir": "img1",
    "frame_rate": 25,
    "seq_length": 750,
    "im_ext": ".jpg"
}

#### [ ] categories

> There are 7 categories used to relate recognitions in the annotations (see anotations).

| **supercategory** | **category** | **id** |
|-------------------|--------------|--------|
| object            | player       | 1      |
| object            | goalkeeper   | 2      |
| object            | referee      | 3      |
| object            | ball         | 4      |
| pitch             | pitch        | 5      |
| camera            | camera       | 6      |
| object            | other        | 7      |

- Every category but **pitch** contains the same properties, while pitch contains all of those, plus **lines** (list of strings).

**Examples**

{
    "supercategory": "object",
    "id": 1,
    "name": "player"
},
{
    ...
},
{
    "supercategory": "pitch",
    "id": 5,
    "name": "pitch",
    "lines": [
        "Big rect. left bottom",
        "Big rect. left main",
        "Big rect. left top",
        "Big rect. right bottom",
        "Big rect. right main",
        "Big rect. right top",
        "Circle central",
        "Circle left",
        "Circle right",
        "Goal left crossbar",
        "Goal left post left",
        "Goal left post right",
        "Goal right crossbar",
        "Goal right post left",
        "Goal right post right",
        "Middle line",
        "Side line bottom",
        "Side line left",
        "Side line right",
        "Side line top",
        "Small rect. left bottom",
        "Small rect. left main",
        "Small rect. left top",
        "Small rect. right bottom",
        "Small rect. right main",
        "Small rect. right top"
    ]
}

#### [ ] images

> One object per frame, containing metadata and labelling.

**Example**

{
    "is_labeled": true,
    "image_id": "1060000001",
    "file_name": "000001.jpg",
    "height": 1080,
    "width": 1920,
    "has_labeled_person": true,
    "has_labeled_pitch": true,
    "has_labeled_camera": true,
    "ignore_regions_y": [],
    "ignore_regions_x": []
}

#### [ ] annotations

> One object per recognized element in each frame, containing its position, category, etc.

##### **"supercategory": "object"**

- **attributes["role"]** specifies the **category** of the recognized object (see categories of supercategory object).
- When **attributes["role"]** equals **referee** or **ball** the properties **jersey** and **team** are null.
- Every **attributes["role"]** equal to **goalkeeper** has its **attributes["jersey"]** equal to null as well (see code below).

In [3]:
from data_analysis import check_goalkeeper_jersey
check_goalkeeper_jersey()

No goalkeeper has jersey number.


**Example**

{
    "id": "3116000275",
    "image_id": "3116000013",
    "track_id": 11,
    "supercategory": "object",
    "category_id": 1,
    "attributes": {
        "role": "player",
        "jersey": "10",
        "team": "right"
    },
    "bbox_image": {
        "x": 1384,
        "y": 433,
        "x_center": 1399.5,
        "y_center": 487.0,
        "w": 31,
        "h": 108
    },
    "bbox_pitch": {
        "x_bottom_left": 50.232524753043776,
        "y_bottom_left": -3.678914519601874,
        "x_bottom_right": 50.64152612344066,
        "y_bottom_right": -3.3791709437732647,
        "x_bottom_middle": 50.43692077833418,
        "y_bottom_middle": -3.5291194334723377
    }
}

##### **"supercategory": "pitch"**

The **pitch** category contains a list of **lines** (see categories section), each of them with a list of 2D coordinates (x,y). Not every line is required.

**Example**

{
    "id": "3116000065",
    "image_id": "3116000003",
    "supercategory": "pitch",
    "category_id": 5,
    "lines": {
        "Side line top": [
            {
                "x": 0.0,
                "y": 0.385512962962963
            },
            {
                "x": 0.12891666666666668,
                "y": 0.37255000000000005
            },
            {
                "x": 0.15856900067318594,
                "y": 0.36927690066767455
            },
            {
                "x": 0.18822133467970523,
                "y": 0.36600380133534904
            },
            {
                "x": 0.24752600269274377,
                "y": 0.35945760267069793
            }
        ],
        "Side line right": [
            {
                "x": 0.2502867772108843,
                "y": 0.3595870370370371
            },
            {
                "x": 0.5432767857142857,
                "y": 0.434142970521542
            },
            {
                "x": 0.8362667942176871,
                "y": 0.5086989040060469
            },
            {
                "x": 1.0,
                "y": 0.5488661438649534
            }
        ],
        "Big rect. right top": [
            {
                "x": 0.04412518069727891,
                "y": 0.4657850718065005
            },
            {
                "x": 0.24620851403061222,
                "y": 0.43862442365835225
            },
            {
                "x": 0.443079970946712,
                "y": 0.41141968379944566
            }
        ],
        "Big rect. right main": [
            {
                "x": 0.04578528911564625,
                "y": 0.4671103552532124
            },
            {
                "x": 0.17843967899659863,
                "y": 0.5152773998488284
            },
            {
                "x": 0.31109406887755103,
                "y": 0.5634444444444444
            },
            {
                "x": 0.5764028486394558,
                "y": 0.6597785336356765
            },
            {
                "x": 1.0,
                "y": 0.8178791194255479
            }
        ],
        "Small rect. right top": [
            {
                "x": 0.6382305484693878,
                "y": 0.4601394494834971
            },
            {
                "x": 0.6093459998582766,
                "y": 0.46550210380448476
            },
            {
                "x": 0.5212104591836734,
                "y": 0.48106254094230266
            }
        ],
        "Small rect. right main": [
            {
                "x": 0.5195965844671202,
                "y": 0.4814357457797934
            },
            {
                "x": 0.7377390323837869,
                "y": 0.5440130605946082
            },
            {
                "x": 0.9584856469671201,
                "y": 0.6056644494834972
            }
        ],
        "Small rect. right bottom": [
            {
                "x": 0.9587283482142858,
                "y": 0.6058549886621315
            },
            {
                "x": 0.9803712957057822,
                "y": 0.6006324829931973
            },
            {
                "x": 1.0,
                "y": 0.5958918367346939
            }
        ],
        "Goal right crossbar": [
            {
                "x": 0.7554180484693878,
                "y": 0.3511872574955908
            },
            {
                "x": 0.8462165727749436,
                "y": 0.3666195105820107
            },
            {
                "x": 0.9404093714569162,
                "y": 0.38326113630637443
            }
        ],
        "Goal right post right": [
            {
                "x": 0.9402430201247166,
                "y": 0.3829644746787604
            },
            {
                "x": 0.9387240097080499,
                "y": 0.45927635739480965
            },
            {
                "x": 0.9372138570011337,
                "y": 0.5330718631897203
            }
        ],
        "Goal right post left": [
            {
                "x": 0.755497941468254,
                "y": 0.3514829931972789
            },
            {
                "x": 0.7550651555413832,
                "y": 0.4152918808264046
            },
            {
                "x": 0.7536190476190475,
                "y": 0.4874026077097506
            }
        ],
        "Circle right": [
            {
                "x": 0.2564324298469388,
                "y": 0.5423689153439153
            },
            {
                "x": 0.25225513392857146,
                "y": 0.5467481670445956
            },
            {
                "x": 0.2470202274659864,
                "y": 0.5525257558578988
            },
            {
                "x": 0.24402543579931973,
                "y": 0.5557664965986394
            },
            {
                "x": 0.24090043579931972,
                "y": 0.5606276077097505
            },
            {
                "x": 0.23894731079931972,
                "y": 0.5647942743764172
            },
            {
                "x": 0.2376452274659864,
                "y": 0.5687294595616024
            },
            {
                "x": 0.23699418579931972,
                "y": 0.5726646447467875
            },
            {
                "x": 0.23734106079931971,
                "y": 0.5759053854875282
            },
            {
                "x": 0.2387301232993197,
                "y": 0.5820785336356765
            },
            {
                "x": 0.24168168579931973,
                "y": 0.5887137188208617
            },
            {
                "x": 0.2475842899659864,
                "y": 0.5955044595616025
            },
            {
                "x": 0.25470251913265307,
                "y": 0.601522978080121
            },
            {
                "x": 0.269111894132653,
                "y": 0.6109359410430839
            },
            {
                "x": 0.2901191857993197,
                "y": 0.6211211262282691
            },
            {
                "x": 0.323799394132653,
                "y": 0.6319239040060468
            },
            {
                "x": 0.36737595663265304,
                "y": 0.6424174225245654
            },
            {
                "x": 0.41911189413265304,
                "y": 0.6516766817838247
            },
            {
                "x": 0.4643379357993197,
                "y": 0.6567692743764172
            },
            {
                "x": 0.5096504357993197,
                "y": 0.6601646447467875
            },
            {
                "x": 0.5793053465136054,
                "y": 0.6625868480725623
            }
        ]
    }
}

##### **"supercategory": "camera"**

> Examples of camera annotations with different (and not null) parameters:

{
    "id": "3116000924",
    "image_id": "3116000042",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": {
        "pan_degrees": 39.77711575201397,
        "tilt_degrees": 82.84357792753083,
        "roll_degrees": -0.970866353352433,
        "position_meters": [
            5.339556045007248,
            40.33155795241939,
            -7.643043960848028
        ],
        "x_focal_length": 3769.055915160941,
        "y_focal_length": 3769.0559151564657,
        "principal_point": [
            960.0,
            540.0
        ],
        "radial_distortion": [
            0.2736005616529989,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0
        ],
        "tangential_distortion": [
            0.0,
            0.0
        ],
        "thin_prism_distortion": [
            0.0,
            0.0,
            0.0,
            0.0
        ]
    },
    "relative_mean_reproj": 0.00775429281319125,
    "accuracy@5": 0.3333333432674408
}

{
    "id": "3116000946",
    "image_id": "3116000043",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": [
        [
            148.05778839202884,
            72.31810812961251,
            -3679.2812380128403
        ],
        [
            3.2816577217872993,
            1.3280349062083199,
            1215.180675602791
        ],
        [
            0.02872126025654286,
            -0.027914925351077088,
            1.0
        ]
    ],
    "relative_mean_reproj": 0.0069903241648980475,
    "accuracy@5": 0.5
}

> List of frames of video 016 whose **has_labeled_camera** property is false
> Observation: **parameters** is null in every case.

In [4]:
from data_analysis import print_camera_annotations
print_camera_annotations()

{
    "id": "3116007220",
    "image_id": "3116000348",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": null,
    "relative_mean_reproj": null,
    "accuracy@5": null
}
{
    "id": "3116007854",
    "image_id": "3116000382",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": null,
    "relative_mean_reproj": null,
    "accuracy@5": null
}
{
    "id": "3116008101",
    "image_id": "3116000397",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": null,
    "relative_mean_reproj": null,
    "accuracy@5": null
}
{
    "id": "3116008503",
    "image_id": "3116000424",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": null,
    "relative_mean_reproj": null,
    "accuracy@5": null
}
{
    "id": "3116008545",
    "image_id": "3116000427",
    "supercategory": "camera",
    "category_id": 6,
    "parameters": null,
    "relative_mean_reproj": null,
    "accuracy@5": null
}
{
    "id": "3116008689",
    "image_id"